# ACV Feature Engineering and Model Experiments

This notebook converts ACV time-series telemetry into car-level features for refrigerant-leakage localisation.

Based on the exploratory analysis, the initial feature set focuses on indoor-temperature behaviour, cooling-performance error, peer-relative behaviour, and persistence of abnormal cooling error.

Each ACV case is represented by one feature row per car. Model evaluation will be performed at the case level to avoid treating correlated timestamps from the same fault case as independent observations.

In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

CODE_ROOT = Path("..").resolve()
REPO_ROOT = Path("../../../..").resolve()

if str(CODE_ROOT) not in sys.path:
    sys.path.append(str(CODE_ROOT))

from src.ingestion import (
    load_acv_file,
    wide_to_long,
)

from src.preprocessing import preprocess_acv
from src.features import extract_car_features

DATA_DIR = REPO_ROOT / "data" / "ACV"

## 1. Build Car-Level Feature Dataset

Each standard-schema training case is passed through the reusable ingestion, preprocessing, and feature-engineering pipeline.

The resulting dataset contains one row for each car, together with its case identifier and ground-truth fault label.

In [3]:
TRAIN_DIR = REPO_ROOT / "data" / "ACV" / "Train"
LABELS_PATH = REPO_ROOT / "data" / "ACV" / "Train_Labels.csv"

labels = pd.read_csv(LABELS_PATH)

labels["faulty_car"] = (
    labels["faulty_car"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

standard_case_names = [
    "acv_case_01.xlsx",
    "acv_case_02.xlsx",
    "acv_case_03.xlsx",
    "acv_case_05.xlsx",
    "acv_case_06.xlsx",
]

feature_tables = []

for filename in standard_case_names:

    df = load_acv_file(TRAIN_DIR / filename)
    df = wide_to_long(df)
    df = preprocess_acv(df)

    features = extract_car_features(df)

    faulty_car = labels.loc[
        labels["filename"] == filename,
        "faulty_car"
    ].iloc[0]

    features["case"] = filename
    features["is_faulty"] = (
        features["car_id"] == faulty_car
    )

    feature_tables.append(features)

feature_df = pd.concat(
    feature_tables,
    ignore_index=True
)

feature_df

,car_id,indoor_temp_mean,indoor_temp_std,cooling_error_mean,cooling_error_median,cooling_error_std,indoor_temp_peer_diff,cooling_error_peer_diff,highest_error_fraction,case,is_faulty
0,01,24.642369,1.682706,0.672225,0.0,1.956908,0.440828,0.712379,0.546030,acv_case_01.xlsx,True
1,02,24.188706,1.665838,0.197960,0.0,1.333551,-0.077644,0.170362,0.189022,acv_case_01.xlsx,False
2,03,24.316514,1.568792,0.032268,0.0,1.102515,0.068422,-0.019000,0.084467,acv_case_01.xlsx,False
3,04,24.245808,1.542009,-0.037567,0.0,1.037490,-0.012384,-0.098812,0.049510,acv_case_01.xlsx,False
4,05,24.150111,1.518204,-0.133265,-0.5,0.994366,-0.121753,-0.208180,0.028788,acv_case_01.xlsx,False
5,06,24.165691,1.553084,-0.117684,-0.5,1.029116,-0.103947,-0.190374,0.033692,acv_case_01.xlsx,False
6,07,24.195903,1.555447,-0.087472,-0.5,1.033732,-0.069419,-0.155846,0.030528,acv_case_01.xlsx,False
7,08,24.148054,1.462062,-0.135321,-0.5,0.931992,-0.124103,-0.210530,0.037963,acv_case_01.xlsx,False
8,01,23.746540,1.158060,0.179790,0.0,0.856418,-0.092600,-0.077956,0.268671,acv_case_02.xlsx,False
9,02,24.038259,1.148649,0.488606,0.5,0.948722,0.240793,0.274977,0.407301,acv_case_02.xlsx,True


### 1.1 Feature Dataset Validation

Verify that each training case contributes exactly eight car-level observations and exactly one labelled faulty car.

In [4]:
feature_validation = (
    feature_df
    .groupby("case")
    .agg(
        cars=("car_id", "nunique"),
        rows=("car_id", "size"),
        faulty_cars=("is_faulty", "sum")
    )
)

feature_validation

,cars,rows,faulty_cars
case,,,
acv_case_01.xlsx,8,8,1
acv_case_02.xlsx,8,8,1
acv_case_03.xlsx,8,8,1
acv_case_05.xlsx,8,8,1
acv_case_06.xlsx,8,8,1


## 2. Leave-One-Case-Out Validation

Because only a small number of independent fault cases are available, randomly splitting individual car observations would allow cars from the same fault case to appear in both the training and validation sets.

Instead, evaluation is performed at the case level using leave-one-case-out validation. In each fold, one complete ACV fault case is held out while the remaining cases form the training set.

This better reflects the final task, where the system must rank cars in a previously unseen fault case.

In [5]:
def rank_decay_score(rank, n_cars):
    """
    Competition rank-decay metric.
    """
    return (n_cars - (rank - 1)) / n_cars

### 2.1 Cooling-Error Rule Baseline

The strongest feature identified during EDA is evaluated using the same case-level ranking procedure that will later be used for machine-learning models.

Within each held-out case, cars are ranked from highest to lowest mean cooling error.

In [6]:
baseline_validation = []

for held_out_case in feature_df["case"].unique():

    test_df = feature_df[
        feature_df["case"] == held_out_case
    ].copy()

    ranked = test_df.sort_values(
        "cooling_error_mean",
        ascending=False
    ).reset_index(drop=True)

    faulty_index = ranked.index[
        ranked["is_faulty"]
    ][0]

    faulty_rank = faulty_index + 1

    score = rank_decay_score(
        faulty_rank,
        len(ranked)
    )

    baseline_validation.append({
        "Held-Out Case": held_out_case,
        "Faulty Car": ranked.loc[
            faulty_index, "car_id"
        ],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["car_id"])
    })

baseline_validation_df = pd.DataFrame(
    baseline_validation
)

baseline_validation_df

,Held-Out Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.0,01|02|03|04|07|06|05|08
1,acv_case_02.xlsx,02,1,1.0,02|03|07|08|06|01|04|05
2,acv_case_03.xlsx,03,1,1.0,03|02|01|07|04|08|06|05
3,acv_case_05.xlsx,04,1,1.0,04|02|07|01|06|03|08|05
4,acv_case_06.xlsx,06,1,1.0,06|08|04|02|03|05|01|07


### 2.2 Baseline Validation Score

The mean rank-decay score across the held-out cases provides the reference score for subsequent modelling experiments.

In [7]:
baseline_loco_score = (
    baseline_validation_df["Score"].mean()
)

print(
    f"Cooling-error baseline score: "
    f"{baseline_loco_score:.3f}"
)

Cooling-error baseline score: 1.000


## 3. Logistic Regression Model

A logistic regression model is used as the first supervised learning approach.

The model combines multiple car-level telemetry features to estimate a fault score for each car. Because the dataset contains substantially more healthy cars than faulty cars, balanced class weights are used.

Evaluation follows the leave-one-case-out strategy. For each fold, the model is trained on four complete fault cases and evaluated on the eight cars belonging to the unseen case.

Features are standardised using statistics calculated only from the training cases to prevent information from the held-out case leaking into model training.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### 3.1 Initial Feature Set

The initial model uses features representing temperature behaviour, cooling performance, peer-relative behaviour, and persistence.

Identifier columns such as car ID and case name are excluded because they do not represent physical ACV behaviour.

In [9]:
model_features = [
    "indoor_temp_mean",
    "indoor_temp_std",
    "cooling_error_mean",
    "cooling_error_median",
    "cooling_error_std",
    "indoor_temp_peer_diff",
    "cooling_error_peer_diff",
    "highest_error_fraction",
]

X = feature_df[model_features]
y = feature_df["is_faulty"].astype(int)

print("Feature matrix:", X.shape)
print("Faulty samples:", y.sum())
print("Healthy samples:", len(y) - y.sum())

Feature matrix: (40, 8)
Faulty samples: 5
Healthy samples: 35


### 3.2 Leave-One-Case-Out Model Evaluation

For each fold, all eight cars from one fault case are excluded from model training.

A pipeline containing `StandardScaler` and `LogisticRegression` is fitted using only the remaining four cases. The trained model then produces a fault probability for each car in the unseen case.

Cars are ranked by predicted fault probability and evaluated using the competition's linear rank-decay metric.

In [10]:
logistic_results = []

for held_out_case in feature_df["case"].unique():

    train_mask = feature_df["case"] != held_out_case
    test_mask = feature_df["case"] == held_out_case

    X_train = feature_df.loc[
        train_mask, model_features
    ]

    y_train = feature_df.loc[
        train_mask, "is_faulty"
    ].astype(int)

    X_test = feature_df.loc[
        test_mask, model_features
    ]

    test_info = feature_df.loc[
        test_mask,
        ["car_id", "is_faulty"]
    ].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    test_info["fault_probability"] = (
        model.predict_proba(X_test)[:, 1]
    )

    ranked = (
        test_info
        .sort_values(
            "fault_probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

    faulty_index = ranked.index[
        ranked["is_faulty"]
    ][0]

    faulty_rank = faulty_index + 1

    score = rank_decay_score(
        faulty_rank,
        len(ranked)
    )

    logistic_results.append({
        "Held-Out Case": held_out_case,
        "Faulty Car": ranked.loc[
            faulty_index, "car_id"
        ],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["car_id"])
    })

logistic_results_df = pd.DataFrame(
    logistic_results
)

logistic_results_df

,Held-Out Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.00,01|02|03|04|07|06|08|05
1,acv_case_02.xlsx,02,1,1.00,02|03|07|01|08|06|04|05
2,acv_case_03.xlsx,03,1,1.00,03|02|01|08|04|07|06|05
3,acv_case_05.xlsx,04,3,0.75,02|01|04|07|06|03|08|05
4,acv_case_06.xlsx,06,1,1.00,06|08|04|02|03|05|01|07


### 3.3 Logistic Regression Validation Score

The mean rank-decay score across all five held-out folds is calculated and compared with the cooling-error baseline.

In [11]:
logistic_loco_score = (
    logistic_results_df["Score"].mean()
)

print(
    f"Cooling-error baseline: {baseline_loco_score:.3f}"
)

print(
    f"Logistic regression:    {logistic_loco_score:.3f}"
)

Cooling-error baseline: 1.000
Logistic regression:    0.950


### 3.4 Observation

The logistic regression model achieved a mean leave-one-case-out rank-decay score of **0.950**, compared with **1.000** for the mean cooling-error rule.

The learned model therefore does not improve upon the simple cooling-error ranking on the available standard-schema cases. This is plausible given the extremely small training set: each validation fold contains only four faulty cars and 28 healthy cars from four independent fault cases.

Furthermore, several engineered features describe related aspects of temperature and cooling behaviour, so combining all features does not necessarily provide additional independent information.

The 1.000 cooling-error baseline should nevertheless not be interpreted as an estimate of perfect unseen-case performance. The feature was identified through exploratory analysis of the same five labelled cases on which it is evaluated. It therefore remains necessary to investigate whether the signal is stable under alternative feature formulations and case-level validation.

## 4. Feature Robustness Analysis

Mean cooling error achieved perfect ranking on the five standard-schema training cases, while the initial logistic regression model achieved a lower leave-one-case-out score.

Because the cooling-error feature was identified using exploratory analysis of these same labelled cases, its perfect observed score may overstate its ability to generalise to an unseen case.

This section therefore evaluates several related fault indicators independently. If the faulty cars remain highly ranked under multiple reasonable definitions of abnormal cooling behaviour, this provides stronger evidence that the observed signal is robust rather than dependent on one particular statistic.

In [12]:
candidate_features = {
    "Mean Cooling Error": "cooling_error_mean",
    "Median Cooling Error": "cooling_error_median",
    "Cooling Error Std": "cooling_error_std",
    "Indoor Temperature Mean": "indoor_temp_mean",
    "Indoor Temperature Peer Difference": "indoor_temp_peer_diff",
    "Cooling Error Peer Difference": "cooling_error_peer_diff",
    "Highest Error Fraction": "highest_error_fraction",
}

ablation_rows = []

for feature_name, feature_column in candidate_features.items():

    case_scores = []

    for case_name, case_df in feature_df.groupby("case"):

        ranked = (
            case_df
            .sort_values(feature_column, ascending=False)
            .reset_index(drop=True)
        )

        faulty_index = ranked.index[
            ranked["is_faulty"]
        ][0]

        faulty_rank = faulty_index + 1

        score = rank_decay_score(
            faulty_rank,
            len(ranked)
        )

        case_scores.append(score)

        ablation_rows.append({
            "Feature": feature_name,
            "Case": case_name,
            "Faulty Car": ranked.loc[
                faulty_index, "car_id"
            ],
            "Faulty Rank": faulty_rank,
            "Score": score
        })

ablation_df = pd.DataFrame(ablation_rows)

### 4.1 Individual Feature Comparison

Each candidate feature is evaluated as an independent ranking score. The table reports its average competition score across the five standard training cases.

In [13]:
feature_comparison = (
    ablation_df
    .groupby("Feature")
    .agg(
        Mean_Score=("Score", "mean"),
        Mean_Faulty_Rank=("Faulty Rank", "mean"),
        Rank_1_Count=(
            "Faulty Rank",
            lambda x: (x == 1).sum()
        )
    )
    .sort_values(
        ["Mean_Score", "Mean_Faulty_Rank"],
        ascending=[False, True]
    )
)

feature_comparison.round(3)

,Mean_Score,Mean_Faulty_Rank,Rank_1_Count
Feature,,,
Cooling Error Peer Difference,1.000,1.0,5
Indoor Temperature Mean,1.000,1.0,5
Indoor Temperature Peer Difference,1.000,1.0,5
Mean Cooling Error,1.000,1.0,5
Highest Error Fraction,0.950,1.4,4
Median Cooling Error,0.875,2.0,3
Cooling Error Std,0.725,3.2,3


### 4.2 Faulty-Car Rank by Case

Average scores can hide case-specific failures. The faulty-car rank for each feature is therefore compared across individual fault cases.

In [14]:
rank_matrix = ablation_df.pivot(
    index="Feature",
    columns="Case",
    values="Faulty Rank"
)

rank_matrix

Case,acv_case_01.xlsx,acv_case_02.xlsx,acv_case_03.xlsx,acv_case_05.xlsx,acv_case_06.xlsx
Feature,,,,,
Cooling Error Peer Difference,1,1,1,1,1
Cooling Error Std,1,1,1,6,7
Highest Error Fraction,1,1,1,3,1
Indoor Temperature Mean,1,1,1,1,1
Indoor Temperature Peer Difference,1,1,1,1,1
Mean Cooling Error,1,1,1,1,1
Median Cooling Error,1,1,3,4,1


### 4.3 Observation

The feature-ablation analysis shows that four individual features rank the labelled faulty car first in all five standard-schema training cases: mean cooling error, cooling-error peer difference, mean indoor temperature, and indoor-temperature peer difference.

These results indicate that the strongest fault-localisation signal is associated with persistently elevated indoor temperature and cooling-performance error rather than increased short-term variability.

The persistence feature remains informative, ranking the faulty car first in four of five cases, but fails to do so in Case 05. Median cooling error and cooling-error standard deviation are less consistent across cases.

Several of the perfectly performing features are mathematically or conceptually related and should therefore not be treated as independent evidence. In particular, peer-difference transformations preserve the within-case ordering of their corresponding mean values.

Consequently, subsequent modelling should prioritise a small set of interpretable temperature and cooling-performance features rather than combining many correlated statistics.

## 5. Case 04 Compatibility Analysis

Case 04 was excluded from the initial feature experiments because it uses a substantially richer telemetry schema than the five standard cases.

The strongest fault-localisation features identified so far depend on indoor temperature and cooling-control temperature. Case 04 contains differently named temperature measurements, including passenger-cabin temperature and target temperature.

Before Case 04 can contribute to the same feature representation, these signals must be inspected to determine whether they provide a meaningful analogue to the standard-schema temperature features.

In [15]:
case_04_raw = load_acv_file(
    TRAIN_DIR / "acv_case_04.xlsx"
)

case_04 = wide_to_long(case_04_raw)

case_04.shape

(178096, 67)

### 5.1 Candidate Temperature Parameters

Inspect the Case 04 parameters that may correspond conceptually to the temperature measurements used in the standard-schema fault features.

In [16]:
case_04_parameters = [
    col
    for col in case_04.columns
    if col != "car_id"
]

candidate_parameters = [
    parameter
    for parameter in case_04_parameters
    if "Temperature" in parameter
]

candidate_parameters

['Target Temperature -2K',
 'Target Temperature -1K',
 'Target Temperature 0',
 'Target Temperature +1K',
 'Target Temperature +2K',
 'Target Temperature Value',
 'Observation Area Temperature Detected Value',
 'Passenger Cabin Temperature Detected Value',
 'Fresh Air Temperature Detected Value']

### 5.2 Temperature Signal Inspection

Case 04 contains `Passenger Cabin Temperature Detected Value` and `Target Temperature Value`, which appear conceptually related to the indoor-temperature and cooling-control measurements used in the standard-schema cases.

Before constructing an equivalent cooling-error feature, the distributions and availability of these measurements are inspected across all eight cars.

In [17]:
case_04_temp_parameters = [
    "Passenger Cabin Temperature Detected Value",
    "Target Temperature Value",
    "Observation Area Temperature Detected Value",
    "Fresh Air Temperature Detected Value",
]

temp_summary_rows = []

for parameter in case_04_temp_parameters:

    if parameter not in case_04.columns:
        continue

    for car_id in sorted(case_04["car_id"].unique()):

        values = case_04.loc[
            case_04["car_id"] == car_id,
            parameter
        ]

        temp_summary_rows.append({
            "Parameter": parameter,
            "Car": car_id,
            "Count": values.notna().sum(),
            "Missing (%)": values.isna().mean() * 100,
            "Mean": values.mean(),
            "Median": values.median(),
            "Min": values.min(),
            "Max": values.max(),
        })

case_04_temp_summary = pd.DataFrame(
    temp_summary_rows
)

case_04_temp_summary.round(2)

,Parameter,Car,Count,Missing (%),Mean,Median,Min,Max
0,Passenger Cabin Temperature Detected Value,01,21849,1.86,25.51,26.0,20.0,28.0
1,Passenger Cabin Temperature Detected Value,02,21849,1.86,25.39,26.0,22.0,29.0
2,Passenger Cabin Temperature Detected Value,03,21849,1.86,25.06,25.0,22.0,29.0
3,Passenger Cabin Temperature Detected Value,04,21842,1.89,25.63,26.0,22.0,29.0
4,Passenger Cabin Temperature Detected Value,05,0,100.00,NaN,NaN,NaN,NaN
5,Passenger Cabin Temperature Detected Value,06,0,100.00,NaN,NaN,NaN,NaN
6,Passenger Cabin Temperature Detected Value,07,0,100.00,NaN,NaN,NaN,NaN
7,Passenger Cabin Temperature Detected Value,08,0,100.00,NaN,NaN,NaN,NaN
8,Target Temperature Value,01,21849,1.86,25.58,26.0,22.0,28.0
9,Target Temperature Value,02,21849,1.86,25.54,26.0,22.0,28.0


### 5.3 Mean Temperature Comparison by Car

The mean value of each candidate temperature signal is compared across cars to determine whether the labelled faulty car exhibits unusual temperature behaviour relative to its peers.

In [18]:
case_04_temp_means = (
    case_04_temp_summary
    .pivot(
        index="Car",
        columns="Parameter",
        values="Mean"
    )
)

case_04_temp_means.round(2)

Parameter,Fresh Air Temperature Detected Value,Observation Area Temperature Detected Value,Passenger Cabin Temperature Detected Value,Target Temperature Value
Car,,,,
01,32.99,25.75,25.51,25.58
02,33.28,-50.00,25.39,25.54
03,33.27,-50.00,25.06,25.22
04,33.27,-50.00,25.63,25.39
05,NaN,NaN,NaN,NaN
06,NaN,NaN,NaN,NaN
07,NaN,NaN,NaN,NaN
08,NaN,NaN,NaN,NaN


### 5.4 Passenger-Cabin Temperature Gap

For the standard-schema cases, the strongest fault-localisation feature was the difference between indoor temperature and cooling-control temperature.

Case 04 does not contain those exact parameters, but it provides passenger-cabin temperature and target temperature measurements. As an exploratory compatibility test, their difference is calculated as a Case-04-specific temperature gap.

This feature is not assumed to be identical to the standard-schema cooling error; it is evaluated only to determine whether it captures a similar relative fault pattern.

In [19]:
case_04_gap = case_04[
    [
        "Time",
        "car_id",
        "Passenger Cabin Temperature Detected Value",
        "Target Temperature Value",
    ]
].copy()

case_04_gap["temperature_gap"] = (
    case_04_gap[
        "Passenger Cabin Temperature Detected Value"
    ]
    - case_04_gap[
        "Target Temperature Value"
    ]
)

case_04_gap_summary = (
    case_04_gap
    .groupby("car_id")
    ["temperature_gap"]
    .agg(
        ["mean", "median", "std", "count"]
    )
    .sort_values(
        "mean",
        ascending=False
    )
)

case_04_gap_summary.round(2)

,mean,median,std,count
car_id,,,,
04,0.24,0.0,0.98,21842
01,-0.08,0.0,0.89,21849
02,-0.15,0.0,0.73,21849
03,-0.16,0.0,0.77,21849
05,NaN,NaN,NaN,0
06,NaN,NaN,NaN,0
07,NaN,NaN,NaN,0
08,NaN,NaN,NaN,0


### 5.5 Observation

The Case-04-specific temperature gap does not reproduce the same fault-localisation pattern observed in the standard-schema cases.

The labelled faulty Car 01 has a mean passenger-cabin-to-target temperature gap of approximately -0.08, while Car 04 has the largest mean gap at approximately 0.24. Ranking the available cars using this feature alone would therefore place the faulty car second rather than first.

Furthermore, valid paired passenger-cabin and target-temperature observations are available only for Cars 01–04. Cars 05–08 contain no observations for this feature combination.

Therefore, `Passenger Cabin Temperature Detected Value - Target Temperature Value` should not be treated as a direct replacement for the standard-schema cooling-error feature. Case 04 requires separate investigation using its richer diagnostic telemetry.

### 5.6 Refrigeration Pressure Signals

Case 04 contains refrigeration-system high- and low-pressure measurements that are unavailable in the standard-schema cases.

Because refrigerant leakage directly affects the refrigeration circuit, these measurements are inspected to determine whether the labelled faulty car exhibits abnormal pressure behaviour relative to the other cars.

This analysis is specific to Case 04 and is not intended to create required features for the final test pipeline, since these pressure measurements are absent from the test-case schema.

In [20]:
pressure_parameters = [
    "Refrigeration System 1 High Pressure Value",
    "Refrigeration System 1 Low Pressure Value",
    "Refrigeration System 2 High Pressure Value",
    "Refrigeration System 2 Low Pressure Value",
]

pressure_summary_rows = []

for parameter in pressure_parameters:

    for car_id in sorted(case_04["car_id"].unique()):

        values = case_04.loc[
            case_04["car_id"] == car_id,
            parameter
        ]

        pressure_summary_rows.append({
            "Car": car_id,
            "Parameter": parameter,
            "Count": values.notna().sum(),
            "Missing (%)": values.isna().mean() * 100,
            "Mean": values.mean(),
            "Median": values.median(),
            "Std": values.std(),
            "Min": values.min(),
            "Max": values.max(),
        })

pressure_summary_df = pd.DataFrame(
    pressure_summary_rows
)

pressure_summary_df.round(2)

,Car,Parameter,Count,Missing (%),Mean,Median,Std,Min,Max
0,01,Refrigeration System 1 High Pressure Value,21849,1.86,1801.17,1860.0,429.69,740.0,2720.0
1,02,Refrigeration System 1 High Pressure Value,21849,1.86,1811.79,1920.0,449.93,780.0,2720.0
2,03,Refrigeration System 1 High Pressure Value,21849,1.86,1806.51,1980.0,522.91,760.0,2860.0
3,04,Refrigeration System 1 High Pressure Value,21842,1.89,2042.13,2100.0,359.72,880.0,2720.0
4,05,Refrigeration System 1 High Pressure Value,0,100.00,NaN,NaN,NaN,NaN,NaN
5,06,Refrigeration System 1 High Pressure Value,0,100.00,NaN,NaN,NaN,NaN,NaN
6,07,Refrigeration System 1 High Pressure Value,0,100.00,NaN,NaN,NaN,NaN,NaN
7,08,Refrigeration System 1 High Pressure Value,0,100.00,NaN,NaN,NaN,NaN,NaN
8,01,Refrigeration System 1 Low Pressure Value,21849,1.86,556.43,500.0,132.67,200.0,1000.0
9,02,Refrigeration System 1 Low Pressure Value,21849,1.86,557.62,480.0,149.76,260.0,1000.0


### 5.7 Mean Refrigeration Pressure by Car

Mean pressure measurements are compared across cars to determine whether the labelled faulty Car 01 exhibits unusual refrigeration-system behaviour.

In [21]:
pressure_mean_matrix = (
    pressure_summary_df
    .pivot(
        index="Car",
        columns="Parameter",
        values="Mean"
    )
)

pressure_mean_matrix.round(2)

Parameter,Refrigeration System 1 High Pressure Value,Refrigeration System 1 Low Pressure Value,Refrigeration System 2 High Pressure Value,Refrigeration System 2 Low Pressure Value
Car,,,,
01,1801.17,556.43,1583.41,568.30
02,1811.79,557.62,1787.83,550.98
03,1806.51,567.88,1771.17,569.40
04,2042.13,441.90,1938.59,407.83
05,NaN,NaN,NaN,NaN
06,NaN,NaN,NaN,NaN
07,NaN,NaN,NaN,NaN
08,NaN,NaN,NaN,NaN


### 5.8 Case 04 Conclusion

The refrigeration-pressure measurements do not provide a straightforward eight-car leakage-localisation feature.

Although Car 01 is the labelled faulty car, Car 04 exhibits the most extreme mean pressure measurements, with higher high-side pressures and lower low-side pressures than Cars 01–03. Therefore, simple pressure magnitude or deviation alone does not identify the labelled refrigerant-leakage car.

In addition, these pressure measurements are unavailable for Cars 05–08, preventing direct comparison across all eight cars.

Combined with the earlier temperature-gap analysis, this indicates that Case 04 cannot be reliably mapped onto the standard-schema fault features using simple parameter substitution. Its substantially different telemetry schema and incomplete parameter availability make it unsuitable for direct pooling with the five standard-schema cases.

Since the held-out test case uses the standard eight-parameter schema, subsequent model development will primarily use Cases 01, 02, 03, 05, and 06, which provide features that can be reproduced consistently during test inference. Case 04 is retained as evidence of schema heterogeneity and motivates dynamic schema handling in the ingestion pipeline.

## 6. Random Forest Model

Random Forest is evaluated as a non-linear supervised alternative to Logistic Regression.

Unlike Logistic Regression, which learns a linear decision boundary, Random Forest can capture non-linear relationships and interactions between the engineered ACV features.

Because only five standard-schema fault cases are available, the model is kept relatively constrained and evaluated using the same leave-one-case-out procedure. This reduces the risk of drawing conclusions from training performance and allows direct comparison with the previous approaches.

In [22]:
from sklearn.ensemble import RandomForestClassifier

### 6.1 Leave-One-Case-Out Evaluation

For each fold, Random Forest is trained using four complete fault cases and evaluated on the eight cars from the remaining unseen case.

The predicted probability of the faulty class is used as the car-level fault score. Cars are ranked from highest to lowest predicted probability and evaluated using the competition's rank-decay metric.

In [23]:
rf_results = []

for held_out_case in feature_df["case"].unique():

    train_mask = feature_df["case"] != held_out_case
    test_mask = feature_df["case"] == held_out_case

    X_train = feature_df.loc[
        train_mask, model_features
    ]

    y_train = feature_df.loc[
        train_mask, "is_faulty"
    ].astype(int)

    X_test = feature_df.loc[
        test_mask, model_features
    ]

    test_info = feature_df.loc[
        test_mask,
        ["car_id", "is_faulty"]
    ].copy()

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    test_info["fault_probability"] = (
        model.predict_proba(X_test)[:, 1]
    )

    ranked = (
        test_info
        .sort_values(
            "fault_probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

    faulty_index = ranked.index[
        ranked["is_faulty"]
    ][0]

    faulty_rank = faulty_index + 1

    score = rank_decay_score(
        faulty_rank,
        len(ranked)
    )

    rf_results.append({
        "Held-Out Case": held_out_case,
        "Faulty Car": ranked.loc[
            faulty_index, "car_id"
        ],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["car_id"])
    })

rf_results_df = pd.DataFrame(rf_results)

rf_results_df

,Held-Out Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.000,01|02|03|04|05|06|07|08
1,acv_case_02.xlsx,02,1,1.000,02|03|01|07|08|04|06|05
2,acv_case_03.xlsx,03,1,1.000,03|02|01|07|08|04|06|05
3,acv_case_05.xlsx,04,2,0.875,02|04|01|07|05|06|03|08
4,acv_case_06.xlsx,06,1,1.000,06|08|01|02|04|03|05|07


### 6.2 Random Forest Validation Score

The mean leave-one-case-out rank-decay score is calculated and compared with the cooling-error baseline and Logistic Regression.

In [24]:
rf_loco_score = rf_results_df["Score"].mean()

print(
    f"Cooling-error baseline: {baseline_loco_score:.3f}"
)

print(
    f"Logistic regression:    {logistic_loco_score:.3f}"
)

print(
    f"Random forest:          {rf_loco_score:.3f}"
)

Cooling-error baseline: 1.000
Logistic regression:    0.950
Random forest:          0.975


## 7. RBF Support Vector Machine

An RBF-kernel Support Vector Machine is evaluated as a second non-linear supervised approach.

The RBF kernel can represent non-linear relationships between the engineered telemetry features without requiring a large model. Feature scaling is performed using only the training portion of each leave-one-case-out fold.

Class weighting is used to account for the imbalance between faulty and healthy cars.

In [25]:
from sklearn.svm import SVC

In [26]:
svm_results = []

for held_out_case in feature_df["case"].unique():

    train_mask = feature_df["case"] != held_out_case
    test_mask = feature_df["case"] == held_out_case

    X_train = feature_df.loc[
        train_mask, model_features
    ]

    y_train = feature_df.loc[
        train_mask, "is_faulty"
    ].astype(int)

    X_test = feature_df.loc[
        test_mask, model_features
    ]

    test_info = feature_df.loc[
        test_mask,
        ["car_id", "is_faulty"]
    ].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)

    # Use decision score directly for ranking.
    test_info["fault_score"] = (
        model.decision_function(X_test)
    )

    ranked = (
        test_info
        .sort_values(
            "fault_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    faulty_index = ranked.index[
        ranked["is_faulty"]
    ][0]

    faulty_rank = faulty_index + 1

    score = rank_decay_score(
        faulty_rank,
        len(ranked)
    )

    svm_results.append({
        "Held-Out Case": held_out_case,
        "Faulty Car": ranked.loc[
            faulty_index, "car_id"
        ],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["car_id"])
    })

svm_results_df = pd.DataFrame(svm_results)

svm_results_df

,Held-Out Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.000,01|02|03|07|06|05|08|04
1,acv_case_02.xlsx,02,1,1.000,02|03|07|01|08|06|04|05
2,acv_case_03.xlsx,03,1,1.000,03|01|02|08|07|04|05|06
3,acv_case_05.xlsx,04,5,0.500,05|02|01|08|04|03|06|07
4,acv_case_06.xlsx,06,4,0.625,08|04|02|06|03|05|01|07


### 7.1 SVM Validation Score

The RBF-SVM leave-one-case-out score is compared with the previous fault-localisation approaches.

In [27]:
svm_loco_score = svm_results_df["Score"].mean()

print(
    f"Cooling-error baseline: {baseline_loco_score:.3f}"
)

print(
    f"Logistic regression:    {logistic_loco_score:.3f}"
)

print(
    f"Random forest:          {rf_loco_score:.3f}"
)

print(
    f"RBF-SVM:                {svm_loco_score:.3f}"
)

Cooling-error baseline: 1.000
Logistic regression:    0.950
Random forest:          0.975
RBF-SVM:                0.825


### 7.2 Model Ranking Comparison

Mean validation score alone does not show how consistently each model localises the faulty car.

The faulty-car rank from each leave-one-case-out fold is therefore compared across the supervised models and the cooling-error scoring rule. This identifies whether lower scores result from isolated near-misses or larger ranking failures.

In [28]:
model_rank_comparison = pd.DataFrame({
    "Case": rf_results_df["Held-Out Case"],
    "Random Forest": rf_results_df["Faulty Rank"],
    "RBF-SVM": svm_results_df["Faulty Rank"],
})

# Add Logistic Regression ranks from the earlier LOCO results
model_rank_comparison["Logistic Regression"] = (
    logistic_results_df["Faulty Rank"].values
)

# Cooling-error rule ranked the faulty car first
# in all five compatible training cases.
model_rank_comparison["Cooling Error"] = 1

model_rank_comparison

,Case,Random Forest,RBF-SVM,Logistic Regression,Cooling Error
0,acv_case_01.xlsx,1,1,1,1
1,acv_case_02.xlsx,1,1,1,1
2,acv_case_03.xlsx,1,1,1,1
3,acv_case_05.xlsx,2,5,3,1
4,acv_case_06.xlsx,1,4,1,1


## 8. Temporal Robustness Analysis

The mean cooling-error feature ranks the labelled faulty car first in all five standard-schema training cases. However, this result is based on aggregating each complete recording into a single mean value.

To determine whether the observed fault signal is persistent rather than being dominated by a limited portion of the recording, each case is divided chronologically into four segments.

Mean cooling error is calculated independently within each segment, and the eight cars are ranked within that segment. A robust fault signal should repeatedly place the labelled faulty car near the top across different portions of the recording.

### 8.1 Load Standard-Schema Cases for Temporal Analysis

The temporal robustness analysis requires the original time-series observations rather than the previously aggregated car-level feature table.

The five training cases compatible with the test schema are therefore loaded and preprocessed individually before being divided into chronological segments.

In [29]:
standard_cases = [
    "acv_case_01.xlsx",
    "acv_case_02.xlsx",
    "acv_case_03.xlsx",
    "acv_case_05.xlsx",
    "acv_case_06.xlsx",
]

training_cases = {}

for case_name in standard_cases:

    raw_df = load_acv_file(
        TRAIN_DIR / case_name
    )

    long_df = wide_to_long(raw_df)

    processed_df = preprocess_acv(long_df)

    # Get the known faulty car from feature_df
    faulty_car = (
        feature_df.loc[
            (feature_df["case"] == case_name)
            & (feature_df["is_faulty"]),
            "car_id"
        ]
        .iloc[0]
    )

    processed_df["is_faulty"] = (
        processed_df["car_id"] == faulty_car
    )

    training_cases[case_name] = processed_df

print("Loaded cases:", list(training_cases.keys()))

for case_name, df in training_cases.items():
    print(
        case_name,
        df.shape,
        "faulty car:",
        df.loc[df["is_faulty"], "car_id"].iloc[0]
    )

Loaded cases: ['acv_case_01.xlsx', 'acv_case_02.xlsx', 'acv_case_03.xlsx', 'acv_case_05.xlsx', 'acv_case_06.xlsx']
acv_case_01.xlsx (50576, 13) faulty car: 01
acv_case_02.xlsx (67056, 13) faulty car: 02
acv_case_03.xlsx (66480, 13) faulty car: 03
acv_case_05.xlsx (50384, 13) faulty car: 04
acv_case_06.xlsx (22520, 13) faulty car: 06


In [30]:
temporal_results = []

for case_name, case_df in training_cases.items():

    case_df = case_df[
        case_df["ACV Information Valid"] == "Valid"
    ].copy()

    case_df["cooling_error"] = (
        case_df["Indoor Average Temperature"]
        - case_df["ACV Control Temperature (Cooling)"]
    )

    unique_times = np.sort(case_df["Time"].unique())

    time_segments = np.array_split(
        unique_times,
        4
    )

    for segment_number, segment_times in enumerate(
        time_segments,
        start=1
    ):

        segment_df = case_df[
            case_df["Time"].isin(segment_times)
        ]

        car_scores = (
            segment_df
            .groupby("car_id")["cooling_error"]
            .mean()
            .sort_values(ascending=False)
        )

        faulty_car = (
            case_df.loc[
                case_df["is_faulty"],
                "car_id"
            ]
            .iloc[0]
        )

        faulty_rank = (
            list(car_scores.index).index(faulty_car) + 1
        )

        temporal_results.append({
            "Case": case_name,
            "Segment": segment_number,
            "Faulty Car": faulty_car,
            "Faulty Rank": faulty_rank,
            "Faulty Score": car_scores.loc[faulty_car],
            "Top Ranked Car": car_scores.index[0],
        })

temporal_results_df = pd.DataFrame(temporal_results)

temporal_results_df

,Case,Segment,Faulty Car,Faulty Rank,Faulty Score,Top Ranked Car
0,acv_case_01.xlsx,1,01,1,0.006009,01
1,acv_case_01.xlsx,2,01,3,0.420936,03
2,acv_case_01.xlsx,3,01,1,-0.299367,01
3,acv_case_01.xlsx,4,01,1,2.592283,01
4,acv_case_02.xlsx,1,02,2,0.331584,03
5,acv_case_02.xlsx,2,02,1,0.412452,02
6,acv_case_02.xlsx,3,02,1,0.525060,02
7,acv_case_02.xlsx,4,02,1,0.689664,02
8,acv_case_03.xlsx,1,03,1,1.093840,03
9,acv_case_03.xlsx,2,03,1,1.173003,03


In [31]:
temporal_rank_matrix = (
    temporal_results_df
    .pivot(
        index="Case",
        columns="Segment",
        values="Faulty Rank"
    )
)

temporal_rank_matrix.columns = [
    f"Segment {i}"
    for i in temporal_rank_matrix.columns
]

segment_columns = [
    "Segment 1",
    "Segment 2",
    "Segment 3",
    "Segment 4",
]

temporal_rank_matrix["Mean Rank"] = (
    temporal_rank_matrix[segment_columns].mean(axis=1)
)

temporal_rank_matrix["Rank 1 Count"] = (
    temporal_rank_matrix[segment_columns]
    .eq(1)
    .sum(axis=1)
)

temporal_rank_matrix

,Segment 1,Segment 2,Segment 3,Segment 4,Mean Rank,Rank 1 Count
Case,,,,,,
acv_case_01.xlsx,1,3,1,1,1.50,3
acv_case_02.xlsx,2,1,1,1,1.25,3
acv_case_03.xlsx,1,1,1,1,1.00,4
acv_case_05.xlsx,2,2,2,1,1.75,1
acv_case_06.xlsx,1,1,1,1,1.00,4


### 8.2 Temporal Robustness Findings

The cooling-error feature remains relatively stable when each recording is divided into four chronological segments.

Across the 20 segment-level evaluations, the labelled faulty car is ranked first in 15 segments. In the remaining five segments, the faulty car remains near the top, with four rank-2 results and one rank-3 result. No segment places the faulty car below rank 3.

Cases 03 and 06 are particularly stable, with the faulty car ranked first in all four segments. Cases 01 and 02 contain isolated segments where the faulty car falls to rank 3 and rank 2 respectively.

Case 05 remains the most challenging case. Its faulty car is ranked second in the first three segments and first only in the final segment. Nevertheless, aggregating the signal across the complete recording ranks the faulty car first.

These results suggest that mean cooling error captures a sustained recording-level difference rather than depending on a single short anomalous period. They also indicate that full-recording aggregation is more reliable for this dataset than requiring the faulty car to exhibit the largest instantaneous cooling error throughout the recording.

## 9. Outlier Robustness Analysis

The full-recording mean cooling error ranks the labelled faulty car first in all five standard-schema cases.

Because an arithmetic mean can be influenced by extreme observations, a 10% trimmed mean is evaluated as a robustness check. For each car, the lowest 10% and highest 10% of cooling-error observations are removed before calculating the mean.

If the faulty-car ranking remains stable after trimming, this provides evidence that the cooling-error result reflects a sustained difference rather than a small number of extreme observations.

In [32]:
from scipy.stats import trim_mean

trimmed_results = []

for case_name, case_df in training_cases.items():

    valid_df = case_df[
        case_df["ACV Information Valid"] == "Valid"
    ].copy()

    valid_df["cooling_error"] = (
        valid_df["Indoor Average Temperature"]
        - valid_df["ACV Control Temperature (Cooling)"]
    )

    faulty_car = (
        valid_df.loc[
            valid_df["is_faulty"],
            "car_id"
        ]
        .iloc[0]
    )

    car_scores = []

    for car_id in sorted(valid_df["car_id"].unique()):

        values = (
            valid_df.loc[
                valid_df["car_id"] == car_id,
                "cooling_error"
            ]
            .dropna()
            .to_numpy()
        )

        car_scores.append({
            "car_id": car_id,
            "mean_score": values.mean(),
            "trimmed_mean_score": trim_mean(
                values,
                proportiontocut=0.10
            )
        })

    scores_df = pd.DataFrame(car_scores)

    mean_ranking = (
        scores_df
        .sort_values("mean_score", ascending=False)
        ["car_id"]
        .tolist()
    )

    trimmed_ranking = (
        scores_df
        .sort_values(
            "trimmed_mean_score",
            ascending=False
        )
        ["car_id"]
        .tolist()
    )

    trimmed_results.append({
        "Case": case_name,
        "Faulty Car": faulty_car,
        "Mean Rank": mean_ranking.index(faulty_car) + 1,
        "Trimmed Mean Rank": trimmed_ranking.index(faulty_car) + 1,
        "Mean Ranking": "|".join(mean_ranking),
        "Trimmed Ranking": "|".join(trimmed_ranking),
    })

trimmed_results_df = pd.DataFrame(trimmed_results)

trimmed_results_df

,Case,Faulty Car,Mean Rank,Trimmed Mean Rank,Mean Ranking,Trimmed Ranking
0,acv_case_01.xlsx,01,1,1,01|02|03|04|07|06|05|08,01|02|03|04|07|08|06|05
1,acv_case_02.xlsx,02,1,1,02|03|07|08|06|01|04|05,02|03|08|07|06|04|01|05
2,acv_case_03.xlsx,03,1,1,03|02|01|07|04|08|06|05,03|02|01|07|08|04|05|06
3,acv_case_05.xlsx,04,1,1,04|02|07|01|06|03|08|05,04|02|07|06|01|03|08|05
4,acv_case_06.xlsx,06,1,1,06|08|04|02|03|05|01|07,06|08|04|02|05|03|01|07


### 9.1 Outlier Robustness Findings

The cooling-error ranking is stable after removing the lowest and highest 10% of observations from each car.

Across all five standard-schema training cases, both the ordinary mean and the 10% trimmed mean rank the labelled faulty car first. Although the ordering of some lower-ranked healthy cars changes, the top-ranked faulty car remains unchanged in every case.

This indicates that the fault-localisation result is not dependent on a small number of extreme cooling-error observations. Instead, the elevated recording-level cooling error appears to be a sustained characteristic of the labelled faulty cars.

## 10. Final ACV Fault-Localisation Method

Based on the feature experiments, supervised-model comparisons and robustness analyses, the final ACV localisation method uses mean cooling error as the car-level fault score.

For each valid telemetry observation:

`cooling_error = Indoor Average Temperature - ACV Control Temperature (Cooling)`

The cooling errors are averaged across the complete recording for each car. Cars are then ranked in descending order of mean cooling error, with a larger value indicating greater sustained cooling underperformance relative to the cooling-control measurement.

This method was selected because it consistently ranked the labelled faulty car first across all five training cases compatible with the test schema. The ranking also remained stable after 10% outlier trimming and generally remained near the top when recordings were evaluated in separate temporal segments.

Three supervised alternatives were evaluated using leave-one-case-out validation. Random Forest, Logistic Regression and RBF-SVM achieved mean rank-decay scores of 0.975, 0.950 and 0.825 respectively, compared with 1.000 for the cooling-error scoring rule on the five compatible labelled cases.

Given the very small number of independent labelled fault cases, the deterministic scoring approach avoids fitting unnecessary model parameters while remaining directly reproducible on the standard-schema test data.

### 10.1 Validation Interpretation

Mean cooling error ranked the labelled faulty car first in all five
schema-compatible training cases. However, this feature was selected after
exploratory analysis of these labelled cases. Therefore, the observed
1.000 rank-decay score should be interpreted as evidence supporting the
chosen fault-localisation rule rather than as an unbiased estimate of
performance on unseen cases.

The supervised models were additionally evaluated using complete
leave-one-case-out splits to reduce leakage between cars from the same
fault case.

## 11. Test-Case Inference

The final fault-localisation method is now applied to the held-out ACV test case.

The test data is processed using the same ingestion and preprocessing pipeline used for the standard-schema training cases. Only telemetry observations marked as valid are used.

For each car, cooling error is calculated as:

`Indoor Average Temperature - ACV Control Temperature (Cooling)`

The mean cooling error across the complete recording is used as the final fault score. Cars are ranked from highest to lowest score.

No test labels are available, and the test data is not used to modify or tune the selected fault-localisation method.

In [33]:
TEST_FILE = REPO_ROOT / "data" / "ACV" / "Test" / "acv_test_case.xlsx"

test_raw = load_acv_file(TEST_FILE)

print("Raw test shape:", test_raw.shape)

test_long = wide_to_long(test_raw)

print("Long test shape:", test_long.shape)

test_processed = preprocess_acv(test_long)

print("Processed test shape:", test_processed.shape)
print("Cars:", sorted(test_processed["car_id"].unique()))

Raw test shape: (9082, 67)
Long test shape: (72656, 12)
Processed test shape: (64112, 12)
Cars: ['01', '02', '03', '04', '05', '06', '07', '08']


### 11.1 Final Car-Level Fault Scores

The frozen cooling-error scoring method is applied independently to each of the eight cars.

Higher mean cooling error indicates that the passenger-cabin temperature remains higher relative to the cooling-control temperature over the recording. Cars are therefore ranked in descending order of this score.

In [34]:
test_valid = test_processed[
    test_processed["ACV Information Valid"] == "Valid"
].copy()

test_valid["cooling_error"] = (
    test_valid["Indoor Average Temperature"]
    - test_valid["ACV Control Temperature (Cooling)"]
)

test_scores = (
    test_valid
    .groupby("car_id")["cooling_error"]
    .agg(
        mean_cooling_error="mean",
        median_cooling_error="median",
        std_cooling_error="std",
        valid_observations="count"
    )
    .reset_index()
)

test_scores = (
    test_scores
    .sort_values(
        "mean_cooling_error",
        ascending=False
    )
    .reset_index(drop=True)
)

test_scores["rank"] = (
    np.arange(1, len(test_scores) + 1)
)

test_scores.round(3)

,car_id,mean_cooling_error,median_cooling_error,std_cooling_error,valid_observations,rank
0,01,0.141,0.0,0.819,8014,1
1,03,-0.026,0.0,0.835,8013,2
2,07,-0.050,0.0,0.852,8012,3
3,04,-0.059,0.0,0.822,7995,4
4,08,-0.070,0.0,0.781,8014,5
5,06,-0.088,0.0,0.815,8013,6
6,02,-0.147,0.0,0.801,8014,7
7,05,-0.166,0.0,0.766,8012,8


### 11.2 Predicted Car Ranking

The eight car identifiers are ordered from highest to lowest mean cooling-error score. This ordering forms the ACV prediction required by the competition submission format.

In [35]:
ranked_cars = test_scores["car_id"].tolist()

ranking_string = "|".join(ranked_cars)

print("Predicted ranking:")
print(ranking_string)

assert len(ranked_cars) == 8
assert len(set(ranked_cars)) == 8
assert set(ranked_cars) == {
    "01", "02", "03", "04",
    "05", "06", "07", "08"
}

Predicted ranking:
01|03|07|04|08|06|02|05


### 11.3 Test Inference Result

Applying the frozen mean cooling-error fault-localisation method to the held-out test case produces the following ranking:

`01|03|07|04|08|06|02|05`

Car 01 receives the highest mean cooling-error score of approximately 0.141, compared with approximately -0.026 for the second-ranked Car 03.

Valid-observation counts are similar across all eight cars, indicating that the ranking is not explained by substantial differences in data availability.

This ranking is retained as the final ACV prediction without further modification of the fault-scoring method based on the test result.

## 12. Production Pipeline Verification

The selected fault-localisation method has been implemented in `src/ACV/pipeline.py`.

The production pipeline is now executed on the same held-out test file to verify that the reusable implementation reproduces the ranking obtained during notebook experimentation.

In [37]:
from src.pipeline import predict_acv

ranked_cars_pipeline, scores_pipeline = predict_acv(
    TEST_FILE
)

scores_pipeline.round(3)

,car_id,fault_score,rank
0,01,0.141,1
1,03,-0.026,2
2,07,-0.050,3
3,04,-0.059,4
4,08,-0.070,5
5,06,-0.088,6
6,02,-0.147,7
7,05,-0.166,8


### 12.1 Pipeline Consistency Check

The production pipeline output is compared with the previously calculated notebook ranking. Both implementations should produce exactly the same ordering of all eight cars.

In [38]:
pipeline_ranking_string = "|".join(
    ranked_cars_pipeline
)

print("Notebook ranking:")
print(ranking_string)

print("\nPipeline ranking:")
print(pipeline_ranking_string)

assert ranked_cars_pipeline == ranked_cars

print("\nPipeline output matches notebook output.")

Notebook ranking:
01|03|07|04|08|06|02|05

Pipeline ranking:
01|03|07|04|08|06|02|05

Pipeline output matches notebook output.
